### Feature Correlation

This section uses a covariance matrix to analyze the correlation between features in the dataset.

In [ ]:
# Load dataset and configurations
import sys
import os
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.datasets import fetch_openml
from sklearn import metrics
import pandas as pd

# Get df and features from data_ingestion
sys.path.append("../Functions")
sys.path.append("../ThirdParty")
from data_ingestion import get_feature_data
X = get_feature_data()
#X = X.iloc[:500]  # use first 500 rows for development

In [ ]:
# Load cached lyrics and create SBERT embeddings (run once; cached to disk)
from genius import Genius, load_lyrics, save_lyrics
from data_ingestion import data_dir
CACHE_PATH = os.path.join(data_dir, "lyrics_cache.pkl")
EMBEDDINGS_PATH = os.path.join(data_dir, "lyric_embeddings.pkl")
client = Genius(os.environ["GENIUS_ACCESS_TOKEN"])  # token required by constructor; embed_text itself is offline
    
if os.path.exists(EMBEDDINGS_PATH):
    embeddings = load_lyrics(EMBEDDINGS_PATH)
    print(f"Loaded {len(embeddings)} embeddings from disk")
else:
    cache = load_lyrics(CACHE_PATH)
    embeddings = {}
    for key, lyrics in cache.items():
        if lyrics is None:
            continue
        try:
            embeddings[key] = client.embed_text(lyrics)
        except Exception as e:
            print(f"Embed failed for {key}: {e}")
    save_lyrics(embeddings, EMBEDDINGS_PATH)
    print(f"Embedded {len(embeddings)} of {len(cache)} cached songs; saved to {EMBEDDINGS_PATH}")

In [ ]:
# Extend classification to include lyric embeddings
import numpy as np
from feature_correlation import *
model = client._get_sbert()
similarities = model.similarity(np.float32(list(embeddings.values())), np.float32(list(embeddings.values())))
print(similarities)

In [ ]:
import numpy as np
import hdbscan
from umap import UMAP

# Stack the dict values into an (n_songs, 384) matrix
keys = list(embeddings.keys())
X_lyrics = np.stack(list(embeddings.values())).astype(np.float32)
print(X_lyrics.shape)  # (500, 384) ish

reducer = UMAP(n_components=15, n_neighbors=30, min_dist=0.0, metric="cosine", random_state=42)
X_lyrics_umap = reducer.fit_transform(X_lyrics)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,        # smallest group worth calling a cluster
    min_samples=1,             # how conservative to be about noise; lower = more clusters
    metric="euclidean",        # safe default for L2-normalized vectors
)
labels = clusterer.fit_predict(X_lyrics_umap)

print(f"clusters found: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"noise points:   {(labels == -1).sum()}")

In [ ]:
clusterer.labels_              # same as labels above
clusterer.probabilities_       # 0..1 confidence per point
clusterer.cluster_persistence_ # how stable each cluster is (higher = more real)

# Map cluster labels back to song keys
from collections import defaultdict
clusters = defaultdict(list)
for key, label in zip(keys, labels):
    clusters[label].append(key)

for cluster_id, songs in sorted(clusters.items()):
    print(f"\nCluster {cluster_id} ({len(songs)} songs):")
    for track, artist in songs[:5]:
        print(f"  {track} — {artist}")

In [ ]:
import plotly.express as px
# size = len(feature_scaling(similarities))
size = 15
keys = list(embeddings.keys())[:size]


fig = px.imshow(similarities)
fig.show()  

In [ ]:
import numpy as np

# Find correlation between audio features
corr = feature_correlation(X)
plot_correlation_matrix(corr)

# Finding the inverse covariance of the scaled features  
VI = inv_cov_matrix(X)

In [ ]:
from umap import UMAP

# Using the covariance matrix to compute the Mahalanobis distance between points
reducer = UMAP(n_components=3,
               n_neighbors=100,
               min_dist=0.1,
               metric="mahalanobis",
               metric_kwds={"VI": VI},
               random_state=42)

audio_matrix = feature_scaling(X)
X_umap = reducer.fit_transform(audio_matrix)
X_umap.shape

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"

fig = px.scatter_3d(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    z=X_umap[:, 2],
    #color=y_named
)

fig.show()